In [ ]:
# Copyright 2026 Google LLC

# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at

#      https://www.apache.org/licenses/LICENSE-2.0

# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# WeatherNext 2 Early Access Program
<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/vertex-ai-samples/blob/main/notebooks/community/weathernext/weathernext_2_ic_early_access_program.ipynb">
      <img src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fvertex-ai-samples%2Fmain%2Fnotebooks%2Fcommunity%weathernext%2Fweathernext_2_ic_early_access_program.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/vertex-ai-samples/blob/main/notebooks/community/weathernext/weathernext_2_ic_early_access_program.ipynb">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/vertexai/v1/32px.svg" alt="Enterprise Gemini Agent Paltform logo"><br> Open in Enterprise Gemini Agent Paltform Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/vertex-ai-samples/blob/main/notebooks/community/weathernext/weathernext_2_ic_early_access_program.ipynb">
      <img width="32px"src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

## Overview

This notebook demonstrates running [WeatherNext 2 inference on Google Cloud Enterprise Gemini Agent Paltform](https://developers.google.com/weathernext/guides/access-vmg) using **customer-provided initial conditions** (custom inputs). WeatherNext 2 is Google's latest medium-range probabilistic forecasting model, principally an operational version the FGN model ([published June 2025](https://arxiv.org/abs/2506.10772)). More information is available in the [WeatherNext documentation](https://developers.google.com/weathernext).

WeatherNext 2 supports customer-provided initial conditions for inference. Instead of using the default ECMWF HRES real-time data, customers can supply their own input Zarr files (e.g., from GFS or their own analysis systems) to generate forecasts with WeatherNext models.

> **Important**: The model is **not fine-tuned** on custom input data. Forecast performance when using custom inputs is **not guaranteed** to match the quality achieved with the default ECMWF HRES inputs. Customers should perform their own evaluation of output quality.

### Objective

- Configure the model inputs for distributed, multi-host inference on H100 or A100 GPUs.
- Provide custom initial conditions (Zarr files) for model inference.
- Run WeatherNext 2 model forecasts in parallel.
- Visualize forecast results.

### Costs

This uses billable components of Google Cloud:

* [Gemini Enterprise Agent Platform]( https://docs.cloud.google.com/gemini-enterprise-agent-platform)
* [Cloud Storage](https://cloud.google.com/storage/docs)
* [Gemini Enterprise Agent Platform Persistent Resource](https://docs.cloud.google.com/gemini-enterprise-agent-platform/machine-learning/training/persistent-resource-create)

Learn about [Gemini Enterprise Agent Platform pricing](https://cloud.google.com/vertex-ai/pricing), [Cloud Storage pricing](https://cloud.google.com/storage/pricing), [Gemini Enterprise Platform Persistent Resource](https://cloud.google.com/products/gemini-enterprise-agent-platform/pricing#custom-trained-models) and use the [Pricing Calculator](https://cloud.google.com/products/calculator/) to generate a cost estimate based on your projected usage.


## Before you begin

### Request For GPU Quota

**WARNING:** Make sure you have sufficient GPU quota allocated for the inference configuration (i.e. `num_samples`) before running Gemini Enterprise Agent Platform Jobs or provisioning a Persistent Resource. Otherwise, some Gemini Enterprise Agent Platform jobs may run while others will fail which would produce
incomplete results.


By default, the quota for GPUs is 0. You can request a higher quota by following the instructions at ["Request a higher quota"](https://cloud.google.com/docs/quota/view-manage#requesting_higher_quota).

You will need to request quota for either **NVIDIA H100 80GB GPUs** or **NVIDIA A100 80GB GPUs** in your selected region. The total number of GPUs you request must be sufficient for your largest planned forecast (i.e., `num_samples`) or your provisioned Persistent Resource capacity.

Depending on whether you use a **Persistent Resource** or run standard custom jobs, you should request the following quota under Service `Enterprise Gemini Agent Paltform API`:

#### Option 1: Persistent Resource
- Name: `Persistent resource Nvidia A100 80GB GPUs per region` OR `Persistent resource Nvidia H100 GPUs per region`

#### Option 2: Standard Custom Model Training (Preemptible)
- Name: `Custom model training preemptible Nvidia A100 80GB GPUs per region` OR `Custom model training preemptible Nvidia H100 GPUs per region`

### Custom Inputs Guide

```
The customer provides input Zarr files in a GCS bucket path via the
`--cns_data_dir` flag. 

#### How to Use

##### Flag: `--input_data_gcs_dir`

`--input_data_gcs_dir` flag for specifying custom input data:

```
--input_data_gcs_dir=gs://customer-bucket/path-to-input-data
```

##### GCS Bucket Setup

We recommend using the **same GCS bucket** for both input data and output
predictions.

The customer should place their input Zarr files under a path within their
existing output bucket, e.g.:

```
gs://customer-bucket/custom-inputs/   ← input Zarr files go here
gs://customer-bucket/outputs/         ← model predictions are written here
```

#### Input File Format

##### Zarr V3 Format Requirement

**All custom input Zarr files MUST be in Zarr V3 format.**

When creating custom input files, ensure they are saved in Zarr V3 format:

```python
import xarray as xa

# When creating input files, ensure they are saved in Zarr V3 format
dataset.to_zarr("path/to/output.zarr", zarr_format=3)
```

If V2 format files are provided as custom input, the job raises
an error when attempting to read them.

##### Example Dataset

Opening an example ECMWF HRES Zarr file with xarray (from
`2026_A1D03180000031800011.zarr`, init time 2026-03-18 00:00 UTC):

```python
>>> import xarray as xa
>>> ds = xa.open_zarr("2026_A1D03180000031800011.zarr")
>>> ds
<xarray.Dataset> Size: ...
Dimensions:  (isobaricInhPa: 25, latitude: 721, longitude: 1440)
Coordinates:
  * isobaricInhPa      (isobaricInhPa) float64 25 ...
  * latitude           (latitude) float64 721 ...
  * longitude          (longitude) float64 1440 ...
    number             int64 ...
    step               int64 ...
    surface            float64 ...
    time               int64 ...
    valid_time         int64 ...
Data variables: (26 total)
    q                  (isobaricInhPa, latitude, longitude) float32 ...
    t                  (isobaricInhPa, latitude, longitude) float32 ...
    u                  (isobaricInhPa, latitude, longitude) float32 ...
    v                  (isobaricInhPa, latitude, longitude) float32 ...
    w                  (isobaricInhPa, latitude, longitude) float32 ...
    z                  (isobaricInhPa, latitude, longitude) float32 ...
    cdir               (latitude, longitude) float32 ...
    cl                 (latitude, longitude) float32 ...
    fdir               (latitude, longitude) float32 ...
    msl                (latitude, longitude) float32 ...
    siconc             (latitude, longitude) float32 ...
    sp                 (latitude, longitude) float32 ...
    ssr                (latitude, longitude) float32 ...
    ssrc               (latitude, longitude) float32 ...
    ssrd               (latitude, longitude) float32 ...
    sst                (latitude, longitude) float32 ...
    t2m                (latitude, longitude) float32 ...
    tcc                (latitude, longitude) float32 ...
    tcwv               (latitude, longitude) float32 ...
    tp                 (latitude, longitude) float32 ...
    u10                (latitude, longitude) float32 ...
    u100               (latitude, longitude) float32 ...
    u200               (latitude, longitude) float32 ...
    v10                (latitude, longitude) float32 ...
    v100               (latitude, longitude) float32 ...
    v200               (latitude, longitude) float32 ...
```

##### Variables

The following variable information is based on the actual contents of
`2026_A1D03180000031800011.zarr`. All data variables are `float32`.

###### Pressure Level Variables (3D: `isobaricInhPa` × `latitude` × `longitude`)

Shape: `[25, 721, 1440]`

| Name | Description (ECMWF short name) |
|------|-------------------------------|
| `z`  | Geopotential                  |
| `t`  | Temperature                   |
| `u`  | U-component of wind           |
| `v`  | V-component of wind           |
| `w`  | Vertical velocity             |
| `q`  | Specific humidity             |

###### Surface Variables (2D: `latitude` × `longitude`)

Shape: `[721, 1440]`

| Name     | Description (ECMWF short name)           |
|----------|------------------------------------------|
| `t2m`    | 2m temperature                           |
| `u10`    | 10m u-component of wind                  |
| `v10`    | 10m v-component of wind                  |
| `u100`   | 100m u-component of wind                 |
| `v100`   | 100m v-component of wind                 |
| `u200`   | 200m u-component of wind                 |
| `v200`   | 200m v-component of wind                 |
| `msl`    | Mean sea level pressure                  |
| `sp`     | Surface pressure                         |
| `sst`    | Sea surface temperature                  |
| `siconc` | Sea ice area fraction                    |
| `tcc`    | Total cloud cover                        |
| `tcwv`   | Total column water vapour                |
| `tp`     | Total precipitation                      |
| `ssr`    | Surface net solar radiation              |
| `ssrc`   | Surface solar radiation clear-sky        |
| `ssrd`   | Surface solar radiation downwards        |
| `cdir`   | Clear-sky direct solar radiation at sfc  |
| `fdir`   | Total sky direct solar radiation at sfc  |
| `cl`     | Low cloud cover                          |

###### Scalar Coordinates

| Name         | dtype     | Description                                |
|--------------|-----------|----------------------------------------------|
| `time`       | `int64`   | Timestamp (units: days since init time, calendar: proleptic_gregorian) |
| `valid_time` | `int64`   | Validity time                              |
| `surface`    | `float64` | Surface level indicator                    |
| `step`       | `int64`   | Forecast step                              |
| `number`     | `int64`   | Ensemble member number                     |

##### Dimension Coordinates

| Coordinate      | dtype     | Shape    | Description                    |
|-----------------|-----------|----------|--------------------------------|
| `isobaricInhPa` | `float64` | `[25]`   | 25 pressure levels in hPa      |
| `latitude`      | `float64` | `[721]`  | 0.25° resolution, 721 points   |
| `longitude`     | `float64` | `[1440]` | 0.25° resolution, 1440 points  |

##### Spatial Resolution

The data is at **0.25° resolution** globally:

- Latitude: 721 points (90°N to 90°S)
- Longitude: 1440 points (0° to 359.75°E)

#### File Naming and Structure

##### File Naming Convention

The inference binary expects input Zarr files to follow a specific file naming
convention. Each file corresponds to a specific forecast initialization time and
uses the following format:

```
<year>_<config><stream><MMDDHHMMMMDDHHMMEE>.zarr
```

Where:

- `<year>`: 4-digit year (e.g., `2025`)
- `<config>`: Data config name (default: `A1`)
- `<stream>`: `D` for 00/12 UTC init times, `S` for 06/18 UTC
- First `MMDDHHMM`: Month, day, hour, minute of the forecast init time
- Second `MMDDHHMM`: Month, day, hour, minute of the validity time
- `EE`: Experiment version (default: `1`)

The validity minute is hardcoded to `01`.

###### Examples

For a forecast initialized at **2026-03-18 00:00 UTC** using fc0:
```
2026_A1D03180000031800011.zarr
```

For a forecast initialized at **2026-03-17 06:00 UTC** using fc0:
```
2026_A1S03170600031706011.zarr
```

##### Success Sentinels

Each Zarr file directory must contain a `success` sentinel file to signal that
the data is complete and ready for reading:

```
gs://<bucket>/custom-inputs/
├── 2026_A1D03180000031800011.zarr/
│   ├── .zmetadata
│   ├── <array data>
│   └── success    ← required sentinel file
├── 2026_A1D03171200031712011.zarr/
│   ├── .zmetadata
│   ├── <array data>
│   └── success
```

The sentinel is a zero-byte file named `success` placed inside each `.zarr`
directory. The job will not proceed with inference until all required input
sentinels exist.

##### Number of Input Files

The model typically requires **2 input timestamps** (the forecast init time and
6 hours prior). For example, for a forecast initialized at 2026-03-18 12:00 UTC,
the binary expects:

1. `2026_A1D03181200031812011.zarr` (init time: 12:00 UTC)
2. `2026_A1D03180600031806011.zarr` (6 hours prior: 06:00 UTC)

#### Caveats and Limitations

1. **No fine-tuning guarantee**: The model is trained on ECMWF HRES
   data. Using custom inputs from a different source may degrade
   forecast quality, especially for features sensitive to the initial
   condition source.

2. **Variable completeness**: All variables listed above must be present in the
   custom input files. Missing variables will cause the inference to fail.

3. **Temporal alignment**: Custom input timestamps must align with valid HRES
   forecast hours (00, 06, 12, or 18 UTC).

4. **File format**: Only Zarr V3 format is accepted.


## Install packages - Restart the kernel after the installation

In [ ]:
# @title Install python packages

# Note that you may need to restart the kernel after this step.
# If so, continue to the next cell after restarting.

print("Installing python packages.")

! pip3 install \
    google-cloud-aiplatform==1.129.0 \
    xarray[complete]

## Authenticate to Google Cloud Platform

In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

## Set Google Cloud Project Pertinent Variables

In [ ]:
# Replace my_gcp_project with your gcp project
PROJECT_ID = "<my_gcp_project>"  # @param {type:"string"}

# Alternatively collect the default cloud project id from the OS env variable
# PROJECT_ID = os.environ["GOOGLE_CLOUD_PROJECT"]

# @markdown 1. [Make sure that billing is enabled for your project](https://cloud.google.com/billing/docs/how-to/modify-project).
# @markdown 2. [Create a Cloud Storage bucket](https://cloud.google.com/storage/docs/creating-buckets) for storing experiment outputs. Set the BUCKET_URI for the experiment environment. The specified Cloud Storage bucket (`BUCKET_URI`) should be located in the same region as where the notebook was launched.

# Replace my_wn_bucket with your bucket
BUCKET_URI = "gs://<my_wn_bucket>"  # @param {type:"string"}

# @markdown 3. Select a region that has the required GPUs available.

# Select a ***US*** region region
REGION = "us-central1"  # @param {type:"string"}


# @markdown 4. Provide the Persistence Resource ID
PERSISTENT_RESOURCE_ID = "<my_persistent_resource_id>"  # @param {type:"string"}

# Set Docker URI
# @markdown 5. Set the Image URI
WEATHERNEXT2_DOCKER_URI = "us-central1-docker.pkg.dev/weathernext-1/wn25-private-preview-launch/weather-next-ic-2-inference.gpu.0-1:latest"

In [ ]:
!gcloud config set project $PROJECT_ID

In [ ]:
# Import the necessary packages
import datetime
import os
import re

from google.cloud import aiplatform

# Enable the Enterprise Gemini Agent Paltform API and Compute Engine API, if not already.
print("Enabling Enterprise Gemini Agent Paltform API and Compute Engine API.")
# ! gcloud services enable aiplatform.googleapis.com compute.googleapis.com

# Cloud Storage bucket for storing the experiment artifacts.
now = datetime.datetime.now().strftime("%Y%m%d%H%M%S")
BUCKET_NAME = "/".join(BUCKET_URI.split("/")[:3])

if BUCKET_URI is None or BUCKET_URI.strip() == "" or BUCKET_URI == "gs://":
    raise ValueError("GCS Bucket URI is invalid!")
else:
    assert BUCKET_URI.startswith("gs://"), "BUCKET_URI must start with `gs://`."
    shell_output = !gcloud storage buckets describe {BUCKET_NAME} --format="value(location)" | tr '[:upper:]' '[:lower:]'
    bucket_region = shell_output[0].strip().lower()
    if bucket_region != REGION:
        raise ValueError(
            f"Bucket region {bucket_region} is different from notebook region {REGION}"
        )
print(f"Using this GCS Bucket: {BUCKET_URI}")

STAGING_BUCKET = os.path.join(BUCKET_URI, "temporal")
# Initialize Enterprise Gemini Agent Paltform API.
aiplatform.init(project=PROJECT_ID, location=REGION, staging_bucket=STAGING_BUCKET)

# Utility functions


def get_job_name_with_datetime(prefix: str) -> str:
    return prefix + datetime.datetime.now().strftime("_%Y%m%d_%H%M%S")

## Configure Model Parameters

In [ ]:
# @markdown ### Hardware Configuration for Distributed Inference
# @markdown - **`machine_type`**: Select a valid machine type. `a3-highgpu` series use NVIDIA H100 80GB GPUs. `a2-ultragpu` series use NVIDIA A100 80GB GPUs.
# @markdown - **`num_samples`**: The total number of ensemble members to generate.
# @markdown The number of machine replicas will be calculated automatically (`num_samples` / GPUs per machine). **Therefore, `num_samples` must be a multiple of the number of GPUs in your selected `machine_type`.**
machine_type = "a2-ultragpu-1g"  # @param ["a3-highgpu-1g", "a3-highgpu-2g", "a3-highgpu-4g", "a3-highgpu-8g", "a2-ultragpu-1g", "a2-ultragpu-2g", "a2-ultragpu-4g", "a2-ultragpu-8g"]
num_samples = 4  # @param {type:"integer"}

# @markdown ### Forecast Configuration
# @markdown - **`forecast_init_time`**: The starting time for the forecast in ISO 8601 format (e.g., `2025-09-21T00:00:00Z`). Models are available for dates from 2024 onwards.
# @markdown - **`horizon_hrs`**: The desired length of the forecast in hours (e.g., 240 for a 10-day forecast).
# @markdown - **`model_seed`**: Choose a specific model seed (1-4) or select "all" to run inference with all four seeds in parallel for improved accuracy.
# @markdown - **`enable_hourly_prediction`**: If checked, the model will generate 1-hour predictions.
forecast_init_time = "2026-05-16T12:00:00Z"

horizon_hrs = 72  # @param {type:"integer"}
model_seed = "all"  # @param ["1", "2", "3", "4", "all"]
enable_hourly_prediction = True  # @param {type:"boolean"}

# --- Parameter Validation and Configuration ---

# Derive accelerator type and count from the chosen machine type
if machine_type.startswith("a3-highgpu"):
    accelerator_type = "NVIDIA_H100_80GB"
elif machine_type.startswith("a2-ultragpu"):
    accelerator_type = "NVIDIA_A100_80GB"
else:
    raise ValueError(f"Invalid machine type selected: {machine_type}.")

try:
    # Extract the number of GPUs from the machine type string, e.g., 'a3-highgpu-4g' -> 4
    accelerators_per_machine = int(re.search(r"-(\d+)g$", machine_type).group(1))
except (AttributeError, ValueError):
    raise ValueError(
        f"Could not determine accelerator count from machine type: {machine_type}"
    )

seeds_to_run = [1, 2, 3, 4] if model_seed == "all" else [int(model_seed)]
num_seeds_to_run = len(seeds_to_run)

num_samples_per_seed = num_samples
if len(seeds_to_run) > 1:
    if num_samples % num_seeds_to_run != 0:
        raise ValueError(
            f"`num_samples` ({num_samples}) is not divisible by the number of seeds to run ({num_seeds_to_run}."
        )
    num_samples_per_seed = num_samples // num_seeds_to_run

# Validate that num_samples is a multiple of accelerators_per_machine
if num_samples_per_seed % accelerators_per_machine != 0:
    raise ValueError(
        f"`num_samples_per_seed` ({num_samples_per_seed}) must be a multiple of the GPUs per machine ({accelerators_per_machine} for {machine_type})."
    )

# Calculate the number of replicas per seed
replica_count_per_seed = num_samples_per_seed // accelerators_per_machine

# Ensure that there are enough samples
if num_samples_per_seed % accelerators_per_machine != 0:
    raise ValueError(
        f"`num_samples_per_seed` ({num_samples_per_seed}) must be a multiple of the GPUs per machine ({accelerators_per_machine} for {machine_type})."
    )

# Calculate total GPUs needed for all jobs
total_gpus_needed = num_samples * (4 if model_seed == "all" else 1)

print("--- Job Configuration Summary ---")
print(f"Total Samples: {num_samples}")
print(f"Machine Type: {machine_type}")
print(f"Accelerator Type: {accelerator_type}")
print(f"GPUs per Machine: {accelerators_per_machine}")
print(f"Total number seeds to run: {num_seeds_to_run}")
print(f"Total number samples per seed: {num_samples_per_seed}")
print(f"Calculated Machine Replicas Per Seed: {replica_count_per_seed}")
print(f"Total GPUs per Job: {num_samples}")
print(f"Total GPUs across all Jobs (ensure sufficient quota): {total_gpus_needed}")
print(f"Docker Image: {WEATHERNEXT2_DOCKER_URI}")
print("---------------------------------")

## Run forecast jobs

In [ ]:
# @markdown This section creates and runs one or more Enterprise Gemini Agent Paltform Custom Training Jobs to generate the forecasts.
# @markdown **This operation is asynchronous.** The jobs will be submitted and this cell will complete quickly.
# @markdown You must monitor the job progress in the Google Cloud Console (https://console.cloud.google.com/vertex-ai/training/custom-jobs).

import time

print(
    f"Submitting {len(seeds_to_run)} job(s) to target persistent resource: {PERSISTENT_RESOURCE_ID}"
)

launched_jobs = []
output_dirs = {}

for seed in seeds_to_run:
    output_dir = os.path.join(BUCKET_URI, "weathernext2_outputs")
    output_dirs[seed] = output_dir

    docker_args_list = [
        f"--pred_root_dir={output_dir}",
        f"--num_samples={num_samples_per_seed}",
        f"--horizon_hrs={horizon_hrs}",
        f"--forecast_init_time={forecast_init_time}",
        f"--model_seed={seed}",
        f"--enable_hourly_prediction={enable_hourly_prediction}",
        f"--input_data_gcs_dir={BUCKET_URI}/custom_inputs/",
    ]

    # Define the worker pool spec for the custom job matching the warm pool specs
    worker_pool_specs = [
        {
            "machine_spec": {
                "machine_type": machine_type,
                "accelerator_type": accelerator_type,
                "accelerator_count": accelerators_per_machine,
            },
            "disk_spec": {
                "boot_disk_type": "pd-standard",
                "boot_disk_size_gb": 100,
            },
            "replica_count": 1,
            "container_spec": {
                "image_uri": WEATHERNEXT2_DOCKER_URI,
                "args": docker_args_list,
            },
        }
    ]

    JOB_NAME = get_job_name_with_datetime(prefix=f"wn2-persistent-run-s{seed}")
    print(f"\n--- Submitting Job for Seed {seed} ---")
    print(f"JOB_NAME: {JOB_NAME}")

    custom_job = aiplatform.CustomJob(
        display_name=JOB_NAME,
        worker_pool_specs=worker_pool_specs,
        project=PROJECT_ID,
        location=REGION,
        staging_bucket=f"{BUCKET_URI}/custom_job_staging",
    )

    # Run the job asynchronously targeting the warm persistent resource pool!
    # Since the 4-GPU persistent resource was created without explicit custom SA runtime permissions,
    # we run using the default Enterprise Gemini Agent Paltform Service Agent (which already has GCS and registry reader permissions).
    custom_job.run(
        persistent_resource_id=PERSISTENT_RESOURCE_ID,
        disable_retries=True,
        sync=False,
    )

    # Wait for GCA resource generation details
    print("Waiting for CustomJob details...")
    for _ in range(15):
        try:
            if custom_job.resource_name:
                break
        except Exception:
            pass
        time.sleep(1)

    job_id = custom_job.name.split("/")[-1]
    print(f"--> Job submitted successfully. ID: {job_id}")
    print(
        f"Console Link: https://console.cloud.google.com/vertex-ai/locations/{REGION}/training/custom-jobs/{job_id}?project={PROJECT_ID}"
    )
    launched_jobs.append(custom_job)

print("\nAll forecast jobs have been submitted. Starting real-time status monitor...")

# Monitoring loop
completed_jobs = set()
while len(completed_jobs) < len(launched_jobs):
    for job in launched_jobs:
        if job.name in completed_jobs:
            continue

        job._sync_gca_resource()
        state = job.state.value if hasattr(job.state, "value") else int(job.state)
        timestamp = datetime.datetime.now().strftime("%H:%M:%S")
        print(
            f"[{timestamp}] Job '{job.display_name}': {job.state.name} (Code: {state})"
        )

        # Final states: 4 is SUCCEEDED, 5 is FAILED, 7 is CANCELLED
        if state in [4, 5, 7]:
            completed_jobs.add(job.name)
            if state == 4:
                print(f"\n🎉 SUCCESS: Job '{job.display_name}' completed successfully!")
            else:
                print(
                    f"\n⚠️ FAILURE/CANCEL: Job '{job.display_name}' exited with code {state}. Error: {getattr(job, 'error', 'N/A')}"
                )

    if len(completed_jobs) < len(launched_jobs):
        time.sleep(20)

print("\nAll forecast runs are complete!")

## Visualize Forecasts (Unified) 

In [ ]:
# @markdown Select which forecast output you want to visualize. This single component
# @markdown can handle both the standard 6-hourly predictions and the datasets
# @markdown with 1-hour model (which have a 'subtime' dimension).
# @markdown If you run into `Error loading Zarr store: unrecognized engine 'zarr'...` try restarting the runtime session and reruning this cell.

# @markdown ---
# @markdown ### Visualization Settings
# @markdown - **`model_seed_to_visualize`**: Choose a specific model seed (1-4) to visualize. This should be one of the model seeds selected in the **Forecast Configuration** above.
# @markdown - **`time_steps_to_visualize`**: Choose to visualize 1-hourly or 6-hourly forecasts. If 1-hourly is selected, ensure `enable_hourly_prediction` was selected in the **Forecast Configuration** above.
# @markdown - **`variable_to_visualize`**: Choose the weather variable to visualize. See the [WeatherNext documentation](https://developers.google.com/weathernext/guides/model-specs-vmg) for variable names and descriptions.
# @markdown - **`sample_to_visualize`**: Choose the sample (ensemble member) to visualize.
# @markdown - **`plot_size`**: Choose the size of the plot to generate.
model_seed_to_visualize = "4"  # @param ["1", "2", "3", "4"]
time_steps_to_visualize = "6-Hourly"  # @param ["6-Hourly", "1-Hourly"]
variable_to_visualize = "2m_temperature"  # @param {type:"string"}
sample_to_visualize = 0  # @param {type:"integer"}
plot_size = 8  # @param {type:"number"}
level_to_visualize = None
# @markdown ---


# --- Helper Functions ---


def init_time_to_folder_path(init_time: str) -> str:
    """
    Convert init time to expected GCS folder path.
    """
    init_date, init_time = init_time.split("T")
    return f"{init_date.replace('-', '')}_{init_time[0:2]}hr"


# override these if you'd like to visualize a different set of forecasts
visualize_bucket = BUCKET_URI
visualize_init_date = forecast_init_time

# set paths based on chosen model seed, bucket, and init date
path_to_6hr_zarr = f"{BUCKET_URI}/weathernext2_outputs/weathernext_2_seed_{model_seed_to_visualize}/{init_time_to_folder_path(visualize_init_date)}_01_preds/predictions.zarr/"
path_to_1hr_zarr = f"{BUCKET_URI}/weathernext2_outputs/weathernext_2_seed_{model_seed_to_visualize}_hourly/{init_time_to_folder_path(visualize_init_date)}_01_preds/predictions.zarr/"


from typing import Optional

import matplotlib
import matplotlib.animation as animation
import matplotlib.pyplot as plt
import numpy as np
import xarray
from IPython.display import HTML

matplotlib.rcParams["animation.embed_limit"] = 500


# --- Helper Functions ---


def select_data(
    data: xarray.Dataset,
    variable: str,
    level: Optional[int] = None,
) -> xarray.Dataset:
    """Selects a variable from the dataset and optionally a level."""
    data = data[variable]
    if "batch" in data.dims:
        data = data.isel(batch=0)
    if level is not None and "level" in data.coords:
        data = data.sel(level=level)
    return data


def scale_data(
    data: xarray.Dataset,
    center: Optional[float] = None,
    robust: bool = False,
) -> tuple[xarray.Dataset, matplotlib.colors.Normalize, str]:
    """Scales the data for visualization."""
    vmin = np.nanpercentile(data.values, (2 if robust else 0))
    vmax = np.nanpercentile(data.values, (98 if robust else 100))
    if center is not None:
        diff = max(vmax - center, center - vmin)
        vmin = center - diff
        vmax = center + diff
    return (
        data,
        matplotlib.colors.Normalize(vmin, vmax),
        ("RdBu_r" if center is not None else "viridis"),
    )


def create_forecast_animation(
    dataset: xarray.Dataset,
    fig_title: str,
    plot_size: float = 5,
    robust: bool = False,
) -> HTML:
    """
    Creates a forecast animation from an xarray Dataset.
    It intelligently handles datasets with or without a 'subtime' dimension.
    """
    # --- Data Preparation ---
    # Check if the data still has 'subtime'). If so, stack dimensions.
    # Otherwise, just rename the 'time' dimension for consistency.
    if "subtime" in dataset.dims:
        print("Detected 'subtime' dimension. Stacking for hourly animation.")
        # Stack 'time' and 'subtime' into a single animation dimension
        plot_data = dataset.stack(animation_step=("time", "subtime")).transpose(
            "animation_step", "lat", "lon"
        )
    else:
        print("No 'subtime' dimension found. Using 'time' for 6-hourly animation.")
        # Use 'time' as the animation dimension
        plot_data = dataset.rename({"time": "animation_step"})

    # Now, the animation dimension is always called 'animation_step'
    max_steps = plot_data.sizes["animation_step"]
    init_time = plot_data.coords["init_time"].values

    # Scale the data for color mapping
    scaled_data, norm, cmap = scale_data(plot_data, robust=robust)

    # --- Plotting Setup ---
    figure = plt.figure(figsize=(plot_size * 2, plot_size))
    ax = figure.add_subplot(1, 1, 1)
    ax.set_xticks([])
    ax.set_yticks([])
    figure.suptitle(fig_title, fontsize=16)
    figure.tight_layout(rect=[0, 0.03, 1, 0.95])  # Adjust for title

    im = ax.imshow(
        scaled_data.isel(animation_step=0), norm=norm, origin="lower", cmap=cmap
    )

    plt.colorbar(
        mappable=im,
        ax=ax,
        orientation="vertical",
        pad=0.02,
        aspect=16,
        shrink=0.75,
        cmap=cmap,
        extend=("both" if robust else "neither"),
    )

    # --- Animation Update Function ---
    def update(frame):
        # Get the coordinates for the current frame
        step_coords = plot_data["animation_step"][frame].coords

        # Calculate total offset and valid time based on available coordinates
        if "subtime" in step_coords:  # Hourly data
            total_offset = step_coords["time"].values + step_coords["subtime"].values
        else:  # 6-hourly data
            total_offset = step_coords["animation_step"].values

        total_hours = total_offset / np.timedelta64(1, "h")
        valid_time = init_time + total_offset
        valid_time_str = np.datetime_as_string(valid_time, unit="m").replace("T", " ")

        new_title = (
            f"{fig_title}\n"
            f"Valid: {valid_time_str} UTC (Forecast: +{total_hours:.1f}h)"
        )
        figure.suptitle(new_title, fontsize=16)
        im.set_data(scaled_data.isel(animation_step=frame))

    # --- Create and Display Animation ---
    ani = animation.FuncAnimation(
        fig=figure, func=update, frames=max_steps, interval=250
    )
    plt.close(figure.number)
    return HTML(ani.to_html5_video())


# --- Main Visualization Logic ---

# 1. Select the correct path based on the user's dropdown choice
if time_steps_to_visualize == "6-Hourly":
    path_to_zarr = path_to_6hr_zarr
elif time_steps_to_visualize == "1-Hourly":
    path_to_zarr = path_to_1hr_zarr
else:
    raise ValueError("Invalid visualization target selected.")

print(f"Loading data from: {path_to_zarr}")

# 2. Load the dataset
try:
    full_dataset = xarray.open_zarr(path_to_zarr)
except Exception as e:
    print(f"Error loading Zarr store: {e}")
    # This is a common point of failure, so we exit gracefully.
else:
    # 3. Select the specific data slice for visualization
    data_for_vis = full_dataset.isel(sample=sample_to_visualize)
    variable_data = select_data(data_for_vis, variable_to_visualize, level_to_visualize)

    # 4. Generate the title
    title = f"{variable_to_visualize} (Sample {sample_to_visualize})"
    if level_to_visualize:
        title += f" at {level_to_visualize} hPa"

    # 5. Create and display the animation
    display(create_forecast_animation(variable_data, title, plot_size, robust=True))